# Cuaderno 02: Análisis Exploratorio de Datos (EDA) - Titanic
Este cuaderno implementa el diagnóstico sobre Bronze y la exploración estadística y visual sobre Silver y Gold.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy import stats

from flows.common.paths import BRONZE, BRONZE_LABELED_NAME, SILVER, SILVER_NAME

# Estilo global
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.titlesize'] = 13

# Carga de Bronze: diagnóstico inicial sobre datos crudos
df_bronze = pd.read_csv(BRONZE / BRONZE_LABELED_NAME)


## Parte 1 — Primer contacto con los datos (Bronze)
### Paso 1: Dimensiones

In [ ]:
print(f"Filas y columnas en Bronze: {df_bronze.shape}")
df_bronze.shape

### Paso 2: Tipos de datos y valores no nulos

In [ ]:
df_bronze.info()

### Paso 3: Primeras 10 filas

In [ ]:
df_bronze.head(10)

### Paso 4: Estadísticas descriptivas

In [ ]:
display(df_bronze.describe())
display(df_bronze.select_dtypes(exclude="number").describe())

### Paso 5: Identificación de datos faltantes en Bronze

In [ ]:
print("Valores nulos por columna:")
print(df_bronze.isnull().sum())

plt.figure(figsize=(9, 4))
sns.heatmap(df_bronze.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Mapa de calor de valores faltantes en Bronze')
plt.show()


## Parte 2 — Conociendo a los pasajeros (Capa Silver)
A partir de aquí consumimos data/silver/titanic_silver.parquet generado de forma determinista y reproducible por flows/01_data_engineering.py.

In [ ]:
df = pd.read_parquet(SILVER / SILVER_NAME)
print(f"Capa Silver cargada con {df.shape[0]} filas y {df.shape[1]} columnas.")
df.head(5)


### Paso 6: Distribución por género y cruce con la clase

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x='Sex', ax=axes[0])
axes[0].set_title('Conteo de pasajeros por género')

sns.countplot(data=df, x='Sex', hue='Pclass', ax=axes[1])
axes[1].set_title('Conteo de pasajeros por género y clase')
plt.tight_layout()
plt.show()

# Proporción de cada clase por género (normalizado por columnas)
pd.crosstab(df['Sex'], df['Pclass'], normalize='columns').round(3)


### Paso 7: Distribución de edades

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df['AgeOriginal'].dropna(), bins=30, kde=True)
plt.title('Distribución de edades conocidas (AgeOriginal)')
plt.xlabel('Edad (años)')
plt.ylabel('Frecuencia')
plt.show()

print(f"Edad promedio: {df['AgeOriginal'].mean():.2f}")
print(f"Mediana: {df['AgeOriginal'].median():.2f}, Mínimo: {df['AgeOriginal'].min():.2f}, Máximo: {df['AgeOriginal'].max():.2f}")


### Paso 8: Variable derivada Person (niño, mujer, hombre)

In [ ]:
print("Distribución de la variable Person:")
print(df['Person'].value_counts())


### Paso 9: Niños por clase

In [ ]:
pd.crosstab(df["Pclass"], df["Person"])

### Paso 10: Gráficos de densidad (KDE) de la edad

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
sns.kdeplot(data=df, x='AgeOriginal', hue='Sex', fill=True, common_norm=False, ax=axes[0])
axes[0].set_title('KDE Edad por Sexo')

sns.kdeplot(data=df, x='AgeOriginal', hue='Person', fill=True, common_norm=False, ax=axes[1])
axes[1].set_title('KDE Edad por Tipo de Persona')

sns.kdeplot(data=df, x='AgeOriginal', hue='Pclass', fill=True, common_norm=False, palette='viridis', ax=axes[2])
axes[2].set_title('KDE Edad por Clase')

plt.tight_layout()
plt.show()


## Parte 3 — Ubicación, embarque y composición familiar
### Paso 11 y 12: Cubierta de la cabina (TCabin)

In [ ]:
print("Conteos por cubierta (incluye X para desconocida):")
print(df['TCabin'].value_counts())

cubierta = df.loc[df['TCabin'] != 'X', 'TCabin']
# Excluyendo la cubierta T anómala solo de la visualización
cubierta_plot = cubierta[cubierta != 'T']

plt.figure(figsize=(7, 4))
sns.countplot(x=cubierta_plot, order=sorted(cubierta_plot.unique()))
plt.title('Distribución de pasajeros por cubierta conocida (sin cubierta T)')
plt.xlabel('Cubierta')
plt.show()


### Paso 13: Puerto de embarque y clase

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(data=df, x='Embarked', hue='Pclass', order=['S', 'C', 'Q'])
plt.title('Pasajeros por puerto de embarque y clase')
plt.show()

# Proporción de clases por puerto (normalizado por filas)
pd.crosstab(df['Embarked'], df['Pclass'], normalize='index').round(3)


### Pasos 14 y 15: Variable Alone (viajar solo vs. con familia)

In [ ]:
print("Proporción de pasajeros solos vs. con familia:")
print(df['Alone'].value_counts(normalize=True).round(3))

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Alone')
plt.title('Distribución de pasajeros solos vs. con familia')
plt.show()


## Parte 4 — Análisis de supervivencia (El núcleo del EDA)
### Paso 16: Panorama general de supervivencia

In [ ]:
print(f"Tasa general de supervivencia: {df['Survived'].mean():.4f}")
sns.countplot(data=df, x='Survivor')
plt.title('Conteo de sobrevivientes vs no sobrevivientes')
plt.show()


### Pasos 17 y 18: Supervivencia por clase y persona

In [ ]:
g1 = sns.catplot(data=df, x='Pclass', y='Survived', kind='point', height=4, aspect=1.3)
g1.ax.set_title('Tasa de supervivencia por clase')
g1.ax.set_ylabel('Tasa de supervivencia')
plt.show()

g2 = sns.catplot(data=df, x='Pclass', y='Survived', hue='Person',
                 hue_order=['child', 'female', 'male'], kind='point', height=4, aspect=1.3)
g2.ax.set_title('Supervivencia por clase y tipo de persona')
g2.ax.set_ylabel('Tasa de supervivencia')
plt.show()


### Paso 19: Relación entre edad y supervivencia

In [ ]:
generaciones = [10, 20, 40, 60, 80]

sns.lmplot(data=df, x='Age', y='Survived', hue='Pclass', x_bins=generaciones, height=4, aspect=1.4)
plt.title('Supervivencia según edad y clase (con bins)')
plt.show()

sns.lmplot(data=df, x='Age', y='Survived', hue='Sex', x_bins=generaciones, height=4, aspect=1.4)
plt.title('Supervivencia según edad y sexo (con bins)')
plt.show()


### Paso 20: Dispersión Tarifa vs. Edad

In [ ]:
plt.figure(figsize=(8, 4))
sns.scatterplot(data=df, x='Age', y='Fare', hue='Pclass', alpha=0.6, palette='viridis')
plt.title('Relación entre Edad y Tarifa según Clase')
plt.xlabel('Edad')
plt.ylabel('Tarifa')
plt.show()


## Parte 5 y 6 — Correlaciones y estadísticas por grupos
### Pasos 25 y 26: Análisis de Títulos extraídos

In [ ]:
print("Frecuencia de títulos consolidados (TTitulo):")
print(df['TTitulo'].value_counts())
print("\nCruce TTitulo x Sex:")
print(pd.crosstab(df['TTitulo'], df['Sex']))


### Paso 27: Matriz de correlación

In [ ]:
cols = ['Survived', 'Pclass', 'NSex', 'Age', 'SibSp', 'Parch', 'Fare', 'NEmbarked', 'NCabin', 'NTitulo']
corr = df[cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Matriz de Correlación de Pearson')
plt.show()

print("Correlaciones directas con Survived ordenadas:")
print(corr['Survived'].sort_values())


### Paso 28 y 29: Estadísticas por grupo y prueba Chi-cuadrado

In [ ]:
tabla_sex = pd.crosstab(df['Sex'], df['Survived'])
chi2, p_val, dof, _ = stats.chi2_contingency(tabla_sex)
print(f"Prueba Chi2 de Sexo vs Supervivencia: chi2 = {chi2:.2f}, p-value = {p_val:.2e}, dof = {dof}")


## Parte 7 — Visualizaciones comparativas avanzadas
### Paso 30: Boxplots múltiples (2x2)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
sns.boxplot(data=df, x='Sex', y='AgeOriginal', ax=axes[0, 0])
axes[0, 0].set_title('Edad por Sexo')

sns.boxplot(data=df, x='Pclass', y='AgeOriginal', ax=axes[0, 1])
axes[0, 1].set_title('Edad por Clase')

sns.boxplot(data=df, x='Embarked', y='AgeOriginal', ax=axes[1, 0])
axes[1, 0].set_title('Edad por Puerto de Embarque')

sns.boxplot(data=df, x='Survived', y='AgeOriginal', ax=axes[1, 1])
axes[1, 1].set_title('Edad por Supervivencia')

plt.tight_layout()
plt.show()


### Paso 31: Distribución y asimetría de la tarifa

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df['Fare'], kde=True, bins=40)
plt.title('Distribución de la Tarifa')
plt.show()

print(f"Media de Fare: {df['Fare'].mean():.2f}")
print(f"Mediana de Fare: {df['Fare'].median():.2f}")
print(f"Asimetría (skewness) de Fare: {df['Fare'].skew():.2f}")


## Parte 9 — Análisis crítico e interacciones (Gold)
### Paso 34: Interacciones Clase x Género

In [ ]:
df.pivot_table(values="Survived", index="Pclass", columns="Sex", aggfunc="mean").round(3)

### Paso 35: Puerto de embarque vs. Clase (Desacople de confusión)

In [ ]:
display(df.groupby('Embarked')['Survived'].agg(['mean', 'count']).round(3))
display(df.pivot_table(values='Survived', index='Embarked', columns='Pclass', aggfunc='mean').round(3))


### Paso 36: Cuartiles de tarifa dentro de cada clase

In [ ]:
df_gold = df.copy()
df_gold['FareQ'] = df_gold.groupby('Pclass')['Fare'].transform(lambda s: pd.qcut(s, 4, labels=False, duplicates='drop'))
df_gold.pivot_table(values='Survived', index='FareQ', columns='Pclass', aggfunc='mean').round(3)


### Paso 37: Tamaño de la familia y supervivencia

In [ ]:
display(df.groupby('Familia')['Survived'].agg(['mean', 'count']))

plt.figure(figsize=(7, 4))
sns.pointplot(data=df, x='Familia', y='Survived')
plt.title('Supervivencia según número de familiares a bordo')
plt.show()


### Paso 38: Pasajeros atípicos (Criterio IQR)

In [ ]:
def outlier_mask_iqr(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return (s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)

for col in ['Fare', 'AgeOriginal', 'Familia']:
    s = df[col].dropna()
    print(f"Atípicos en {col}: {outlier_mask_iqr(s).sum()}")

fare_mask = outlier_mask_iqr(df['Fare'])
print("\nSupervivencia de atípicos de Fare vs. no atípicos:")
print(df.groupby(fare_mask)['Survived'].agg(['mean', 'count']))

print("\nTop 10 tarifas más altas registradas:")
df.nlargest(10, 'Fare')[['Name', 'Pclass', 'Fare', 'Survived']]
